In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

np.random.seed(42)



In [2]:
train_path = "../input/plant-pathology-2021-fgvc8/train.csv"
train_df = pd.read_csv(train_path)

most_common_label = train_df["labels"].value_counts().idxmax()

all_labels = train_df["labels"].str.split(" ").explode().unique()
label2idx = {lab: i for i, lab in enumerate(all_labels)}
freq = np.zeros(len(all_labels), dtype=float)

for lab_list in train_df["labels"].str.split(" "):
    for lab in lab_list:
        freq[label2idx[lab]] += 1
freq /= len(train_df)



In [3]:
train_split, val_split = train_test_split(
    train_df, test_size=0.2, random_state=42, shuffle=True
)


def multilabel_binarize(label_series):
    """Convert a Series of space‑delimited label strings to a binary matrix."""
    bin_mat = np.zeros((len(label_series), len(all_labels)), dtype=int)
    for i, labs in enumerate(label_series.str.split(" ")):
        for lab in labs:
            bin_mat[i, label2idx[lab]] = 1
    return bin_mat


val_truth = multilabel_binarize(val_split["labels"])


def predict_with_threshold(th):
    """Assign every label whose training frequency ≥ th."""
    pred = (freq >= th).astype(int)  # shape (n_labels,)
    pred_mat = np.tile(pred, (len(val_split), 1))
    if not pred.any():
        most_idx = label2idx[most_common_label]
        pred_mat[:, most_idx] = 1
    return pred_mat


best_thr = 0.0
best_f1 = -1.0
# finer search (step 0.01) to get a slightly better validation F1
for thr in np.arange(0.0, 1.001, 0.01):
    val_pred = predict_with_threshold(thr)
    f1 = f1_score(val_truth, val_pred, average="micro")
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

if not (freq >= best_thr).any():
    best_thr = 0.0  # fallback to most common label later



In [4]:
# Build a robust path to the test images folder
TEST_FOLDER = os.path.abspath("../input/plant-pathology-2021-fgvc8/test_images/")
if not os.path.isdir(TEST_FOLDER):
    raise RuntimeError(f"Test folder not found: {TEST_FOLDER}")

# List only image files (ignore hidden files)
test_images = sorted(
    [
        f
        for f in os.listdir(TEST_FOLDER)
        if f.lower().endswith((".jpg", ".png", ".jpeg"))
    ]
)
submission = pd.DataFrame(test_images, columns=["image"])

selected_labels = [lab for lab, f in zip(all_labels, freq) if f >= best_thr]
if not selected_labels:
    submission["labels"] = most_common_label
else:
    # join multiple labels with a space as required by the competition
    submission["labels"] = " ".join(selected_labels)



In [5]:
# Write submission to the current working directory (Kaggle expects this name)
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path} with {len(submission)} rows.")

Submission file written to submission.csv with 3727 rows.
